<a href="https://colab.research.google.com/github/jetnipitbank-maker/thai-bank-sentiment-analysis/blob/main/03_domain_adaptive_pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL-01]
!pip install -q transformers[torch] datasets pythainlp pandas-gbq accelerate sentencepiece

In [ ]:
import os
import glob
import json
import re
from pythainlp.util import normalize
from tqdm import tqdm

def clean_finance_text(text):
    if not text: return ""
    # 1. Normalize สระ/วรรณยุกต์ไทย
    text = normalize(text)
    # 2. ลบ URL และอีเมล
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    # 3. ลบ HTML tags
    text = re.sub(r'<.*?>', '', text)
    # 4. ลบขยะท้ายข่าว
    garbage_patterns = [
        r'อ่านข่าวที่เกี่ยวข้อง.*', r'รายงานโดย.*', r'เรียบเรียงโดย.*',
        r'เว็บไซต์นี้มีการใช้งานคุกกี้.*', r'ติดตามความเคลื่อนไหว.*',
        r'สนใจสมัครสมาชิก.*', r'สอบถามข้อมูลเพิ่มเติม.*'
    ]
    for pattern in garbage_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    # 5. ลบช่องว่างส่วนเกิน
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def run_preprocessing_multi_files(input_folder, output_file):
    print(f"📂 กำลังค้นหาไฟล์ในโฟลเดอร์: {input_folder}")
    jsonl_files = glob.glob(os.path.join(input_folder, '*.jsonl'))

    if not jsonl_files:
        print("⚠️ ไม่พบไฟล์ .jsonl ในโฟลเดอร์ที่ระบุ")
        return

    print(f"📁 พบ {len(jsonl_files)} ไฟล์ .jsonl")

    final_results = []
    total_news_processed = 0

    for file_path in jsonl_files:
        current_file_count = 0
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in tqdm(f, desc=f"คลีน {os.path.basename(file_path)}"):
                if not line.strip(): continue
                try:
                    item = json.loads(line)
                    # ใช้คีย์ 'content' เป็นหลักตามที่ตรวจสอบพบ
                    title = item.get('title', '')
                    content = item.get('content', '')
                    full_content = f"{title} {content}".strip()

                    if not full_content:
                        continue

                    cleaned = clean_finance_text(full_content)

                    if len(cleaned) > 50:
                        final_results.append(cleaned)
                        total_news_processed += 1
                        current_file_count += 1
                except Exception as e:
                    print(f"Error in {os.path.basename(file_path)}: {e}")
                    continue
        print(f"✅ ประมวลผลเสร็จ: {os.path.basename(file_path)} (ได้ {current_file_count} ข่าว)")

    print(f"💾 กำลังบันทึกไฟล์ {total_news_processed:,} ข่าว ไปที่: {output_file}")
    with open(output_file, 'w', encoding='utf-8') as f:
        for line in final_results:
            f.write(line + "\n")
    print("✨ รวมไฟล์เสร็จสมบูรณ์!")

# รันการประมวลผล
INPUT_FOLDER = r"C:\Users\USER\Desktop\News_DAPT\Final_Data"
OUTPUT_TXT_COMBINED = "dapt_corpus_combined.txt"
run_preprocessing_multi_files(INPUT_FOLDER, OUTPUT_TXT_COMBINED)

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

DATASET_PATH = r"C:\Users\USER\Desktop\News_DAPT\Final_Data\dapt_corpus_combined.txt"
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"
OUTPUT_DIR = "./wangchanberta-finance-dapt"

print("⏳ 1. กำลังโหลด Dataset...")
dataset = load_dataset("text", data_files={"train": DATASET_PATH})

print("⏳ 2. กำลังโหลด Tokenizer และ Model (แบบดั้งเดิม/เสถียร)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

# ไม่ต้อง resize vocab แล้ว เพราะเราไม่ได้เพิ่ม token ใหม่
# model.resize_token_embeddings(len(tokenizer))

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=400, # ความยาวปลอดภัย
        return_token_type_ids=False
    )

print("⏳ 3. กำลัง Tokenize ข้อมูล 1.6 แสนข่าว...")
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("⏳ 4. กำลังเตรียม Data Collator สำหรับ MLM...")
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

print("⏳ 5. ตั้งค่าตัวแปรสำหรับ Training (โหมด Safe Mode)...")
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    # 🌟 ปิดระบบเร่งความเร็วทั้งหมด ให้การ์ดจอคิดแบบเต็มความละเอียด (fp32)
    fp16=False,
    bf16=False,
    save_steps=2000,
    save_total_limit=1,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"],
)

print("\n🔥 =============================== 🔥")
print("🚀 เริ่มเทรน Domain Adaptive Pre-training (DAPT)!")
print("🔥 =============================== 🔥\n")

# บังคับล้างขยะ VRAM ก่อนเริ่ม
torch.cuda.empty_cache()
trainer.train()

print("\n💾 กำลังบันทึกโมเดลสุดท้ายที่สมบูรณ์...")
trainer.save_model(f"{OUTPUT_DIR}-final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}-final")

print("✅ เทรนเสร็จสมบูรณ์ 100% ไม่มีบั๊กเจือปน!")

In [ ]:
import math
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

test_sentences = [
    "ธนาคารแห่งประเทศไทย (ธปท.) ชี้แจงแนวโน้มเงินเฟ้อพื้นฐานที่ยังคงทรงตัว",
    "นักลงทุนสถาบันเทขายหุ้นในกลุ่มพลังงานและธนาคารพาณิชย์กดดันดัชนี SET",
    "อัตราผลตอบแทนพันธบัตรรัฐบาลสหรัฐอายุ 10 ปี ปรับตัวสูงขึ้นแตะระดับสูงสุด",
    "บริษัทเตรียมจ่ายเงินปันผลระหว่างกาลในอัตรา 0.50 บาทต่อหุ้น",
    "หนี้ที่ไม่ก่อให้เกิดรายได้ (NPL) ของระบบธนาคารพาณิชย์มีแนวโน้มเพิ่มขึ้นเล็กน้อย",
    "ก.ล.ต. สั่งเปรียบเทียบปรับผู้บริหารฐานอินไซเดอร์เทรดดิ้ง",
    "หุ้น IPO ของบริษัทเทคโนโลยีได้รับความสนใจจากนักลงทุนรายย่อยอย่างล้นหลาม",
    "ค่าเงินบาทอ่อนค่าลงทะลุแนวต้านสำคัญเมื่อเทียบกับดอลลาร์สหรัฐ",
    "กำไรสุทธิของบริษัทจดทะเบียนในไตรมาส 3 เติบโตโดดเด่นเหนือความคาดหมาย",
    "มาตรการกระตุ้นเศรษฐกิจของภาครัฐคาดว่าจะช่วยฟื้นฟูความเชื่อมั่นผู้บริโภค"
] * 10

test_dataset = Dataset.from_dict({"text": test_sentences})

models_to_test = {
    "1. Base Model (ก่อนเทรน)": "airesearch/wangchanberta-base-att-spm-uncased",
    "2. DAPT Model (หลังเทรน)": "./wangchanberta-finance-dapt-final"
}

results = {}

# กลับมาใช้การ์ดจอสุดแรงของเรา
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ อุปกรณ์ที่ใช้รัน: {device}")
print("🔥 เริ่มการทดสอบวัดระดับความงง (Perplexity) 🔥\n")

for model_name, model_path in models_to_test.items():
    print(f"=========================================")
    print(f"⏳ กำลังประเมิน: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
    model = AutoModelForMaskedLM.from_pretrained(model_path, attn_implementation="eager")
    model.to(device)
    model.eval()

    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=400, return_token_type_ids=False)

    tokenized_eval = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

    dataloader = DataLoader(tokenized_eval, batch_size=8, collate_fn=data_collator)

    total_loss = 0.0
    total_batches = 0
    vocab_size = model.config.vocab_size
    unk_id = tokenizer.unk_token_id if tokenizer.unk_token_id is not None else 0

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}

            # 🌟🌟🌟 ระบบเกราะป้องกันขั้นสูงสุด: กรอง Token ที่ทะลุพจนานุกรม 🌟🌟🌟
            # ถ้าเจอ ID ที่ใหญ่กว่าพจนานุกรม ให้เปลี่ยนเป็น <unk>
            batch["input_ids"][batch["input_ids"] >= vocab_size] = unk_id

            if "labels" in batch:
                out_of_bounds = (batch["labels"] >= vocab_size) & (batch["labels"] != -100)
                batch["labels"][out_of_bounds] = unk_id
            # -----------------------------------------------------------------

            outputs = model(**batch)
            total_loss += outputs.loss.item()
            total_batches += 1

    avg_loss = total_loss / total_batches
    perplexity = math.exp(avg_loss)
    results[model_name] = perplexity
    print(f"🎯 Loss: {avg_loss:.4f}  👉  Perplexity: {perplexity:.2f}\n")

    del model
    torch.cuda.empty_cache()

print("\n🏆 ================= สรุปผลการทดสอบ ================= 🏆")
for name, ppl in results.items():
    print(f"{name:<25} : PPL = {ppl:.2f}")

base_ppl = results["1. Base Model (ก่อนเทรน)"]
dapt_ppl = results["2. DAPT Model (หลังเทรน)"]
improvement = ((base_ppl - dapt_ppl) / base_ppl) * 100

print("-" * 55)
if improvement > 0:
    print(f"✨ ยินดีด้วย! โมเดลของคุณมีความงงลดลง (เก่งขึ้น) ถึง : {improvement:.2f}% 🚀")
else:
    print(f"🤔 ความงงเพิ่มขึ้นเล็กน้อย")
print("==========================================================")